<b>Salary Survery Clean and Wrangle</b>

<b>Guide:</b>
- Load and inspect the data; profile it (shape, dtypes, missingness, duplicates)
- Diagnose specific data quality issues present in your dataset — don't apply a generic checklist blindly
- Choose and justify cleaning strategies (deletion vs. imputation, and which imputation method, if any) —
explain trade-offs, not just what you did
- Fix data types, remove/handle duplicates, handle outliers deliberately (don't silently drop them without
justification)
- Merge/join/concatenate and reshape (melt/pivot) as needed to build one analysis-ready table
- Handle formatting/encoding issues (e.g., inconsistent categories, string casing, encoding errors) if
present
- Document every decision in markdown cells: what the issue was, what you did about it, and why — this
log is graded


# Load and Inspect

In [239]:
import os
import numpy as np
import pandas as pd
import missingno as msno

df = pd.read_csv('../Dataset/Salary Survey 2021.csv')
df_raw = df.copy()

In [240]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28225 entries, 0 to 28224
Data columns (total 18 columns):
 #   Column                                                                                                                                                                                                                                Non-Null Count  Dtype  
---  ------                                                                                                                                                                                                                                --------------  -----  
 0   Timestamp                                                                                                                                                                                                                             28225 non-null  str    
 1   How old are you?                                                                                                        

## Functions

### print_shape

In [241]:
def print_shape(option="both"):
    df_raw_rows, df_raw_cols = df_raw.shape
    df_rows, df_cols = df.shape
    
    option = option.lower()
    
    if option == "row" or option == "both":
        print(f"Total Rows: {df_rows}/{df_raw_rows}")
        
    if option == "col" or option == "both":
        print(f"Total Columns: {df_cols}/{df_raw_cols}")

print_shape("col")

Total Columns: 18/18


### print_unique_summary

In [242]:
def print_unique_summary(df_name, column_name):
    unique_vals = df_name[column_name].unique()
    num_unique = df_name[column_name].nunique()

    print(f"Total Unique Count (nunique): {num_unique}")
    print(f"Unique Values (unique): {unique_vals}\n")

### map_category
<b>Description: </b> Used for Keyword Matching Data Cleaning

In [243]:
def map_category(value, mapping):
    if pd.isna(value):
        return value
    for category, pattern in mapping.items():
        if pd.Series([value]).str.contains(pattern, case=False, na=False).iloc[0]:
            return category
    return "Other"

### print_null_sum

In [244]:
def print_null_sum(df_name, column_name):
    print("Total NULL Values:", df_name[column_name].isnull().sum())

# Column Cleaning

In [245]:
df.columns.to_list()

['Timestamp',
 'How old are you?',
 'What industry do you work in?',
 'Job title',
 'If your job title needs additional context, please clarify here:',
 "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)",
 'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.',
 'Please indicate the currency',
 'If "Other," please indicate the currency here: ',
 'If your income needs additional context, please provide it here:',
 'What country do you work in?',
 "If you're in the U.S., what state do you work in?",
 'What city do you work in?',
 'How many years of professional work experience do you have overall?',
 'How many years of professional work experience do you have in your field?',

## Remove unnecessary Fields 
- 'Timestamp',
- 'If your job title needs additional context, please clarify here:',
- 'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.',
- 'If "Other," please indicate the currency here: ',
- 'If your income needs additional context, please provide it here:',
- "If you're in the U.S., what state do you work in?",
- 'What city do you work in?',
- 'How many years of professional work experience do you have overall?'
- 'What is your race? (Choose all that apply.)'

In [246]:
df = df.drop(columns=[
    'Timestamp',
    'If your job title needs additional context, please clarify here:',
    'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.',
    'If "Other," please indicate the currency here: ',
    'If your income needs additional context, please provide it here:',
    "If you're in the U.S., what state do you work in?",
    'What city do you work in?',
    'How many years of professional work experience do you have overall?',
    'What is your race? (Choose all that apply.)'
])

In [247]:
print_shape()

Total Rows: 28225/28225
Total Columns: 9/18


## Rename needed field
- age = 'How old are you?',
- industry = 'What industry do you work in?',
- job_title 'Job title',
- annual_salary = "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)",
- currency 'Please indicate the currency',
- country 'What country do you work in?',
- total_years_experience = 'How many years of professional work experience do you have overall?',
- current_years_experience 'How many years of professional work experience do you have in your field?',
- education 'What is your highest level of education completed?',
- gender 'What is your gender?',

In [248]:
df = df.rename(columns={
    'How old are you?' : 'age_range',
    'What industry do you work in?' : 'industry',
    'Job title' : 'job_title',
    "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)" : 'annual_salary',
    'Please indicate the currency' : 'currency',
    'What country do you work in?' : 'country',
    'How many years of professional work experience do you have in your field?' : 'current_years_experience',
    'What is your highest level of education completed?' : 'education',
    'What is your gender?' : 'gender'
})

In [249]:
print_shape()
df.columns.tolist()

Total Rows: 28225/28225
Total Columns: 9/18


['age_range',
 'industry',
 'job_title',
 'annual_salary',
 'currency',
 'country',
 'current_years_experience',
 'education',
 'gender']

## Cleaning Log
- The dataset originally had 18 columns, but only 10 remain since these are needed for analysis.  
- Some column names are too long, so need to be renamed for easy analysis and data rows cleaning.  
- Unnecessary columns have been removed.  

# Data Diagnosis
<b>Description:</b> 
- Inspect data for duplicate category, redundancy, etc
- use for data cleaning testing (use <code>df_temp</code> as data frame tester))

In [250]:
df.columns.to_list()

['age_range',
 'industry',
 'job_title',
 'annual_salary',
 'currency',
 'country',
 'current_years_experience',
 'education',
 'gender']

## age_range

In [251]:
print_unique_summary(df, "age_range")
df["age_range"].isnull().sum()

Total Unique Count (nunique): 7
Unique Values (unique): <StringArray>
['25-34', '45-54', '35-44', '18-24', '65 or over', '55-64', 'under 18']
Length: 7, dtype: str



np.int64(0)

### Cleaning Logs

**Status:** No cleaning needed

- No null values found
- No invalid/duplicate age ranges found

## industry

In [252]:
print_unique_summary(df, "industry")
print("Total NULL Values:", df["industry"].isnull().sum())

# 1225 total unique values. There are lot of redundant categories for industry e.g. (Software Development, Software/Programming) it could be put in one group like Information Technology as general
# Remove duplicate categories as much as possible.
# 86 NULL Values could be included as Other Industry
# Do Fuzzy Matching or  keyword matching
# use df_temp for testing.

#after data cleaning 22 unique values

Total Unique Count (nunique): 1225
Unique Values (unique): <StringArray>
[                'Education (Higher Education)',
                            'Computing or Tech',
                'Accounting, Banking & Finance',
                                   'Nonprofits',
                                   'Publishing',
                'Education (Primary/Secondary)',
                                          'Law',
                                  'Health care',
               'Utilities & Telecommunications',
                       'Business or Consulting',
 ...
                                     'Plumbing',
 'I'm currently a student and don't have a job',
                                     'Student ',
                               'Wine & Spirits',
                              'Social networks',
                                          'ABA',
                                   'psychology',
                                       'School',
                                    'lif

In [253]:
# Keyword Matching Data Cleaning
#\bcall\b exact word "call"
df_temp = df.copy()
industry_mapping = {
    "Contact Center Services": r"\bcall\b|contact|customer service|\bbpo\b|business process|customer",
    "Education": r"teacher|educ|school|university|acade|phd|ed ?tech",
    "Health": r"health|medical|hospital|nursing|pharma|\bcare\b|psycho",
    "Biotech": r"biotech|bitech|pharma",
    "Finance": r"bank|finance|accounting|insurance|investment",
    "Legal": r"law|legal|attorney",
    "Government": r"government|public admin|civil service|military|govt",
    "Nonprofit": r"nonprofit|non-profit|ngo|charity",
    "Marketing": r"market|adverti|public relations",
    "Retail/Sales": r"retail|\bsale\b|e-commerce",
    "Engineering/Manufacturing": r"manufactur|engineer|construction|industrial",
    "Media/Entertainment": r"media|publish|journal|entertainment|film|music",
    "Hospitality": r"hospitality|restaurant|hotel|\bfood\b|\bchild\b|coffee",
    "Arts/Design": r"\bart\b|design|creative",
    "Real Estate": r"real estate|property",
    "Transportation": r"transport|logistics|shipping|airline",
    "Agriculture": r"agricult|\bfarm\b",
    "Energy": r"energy|\boil\b|\bgas\b|utilit",
    "Insurance": r"insurance",
    "Vet": r"vet|animal|\bcat\b|\bdog\b",
    "Business/Consulting": r"business|consult",
    "Information Technology": r"software|\bit\b|comput|\bdata\b|technology|information|cyber",
}

df_temp["industry"] = df_temp["industry"].apply(lambda x: map_category(x, industry_mapping))

In [254]:
print_unique_summary(df_temp, "industry")
df_temp.industry.value_counts()

Total Unique Count (nunique): 22
Unique Values (unique): <StringArray>
[                'Education',    'Information Technology',
                   'Finance',                 'Nonprofit',
       'Media/Entertainment',                     'Legal',
                    'Health',                    'Energy',
       'Business/Consulting',               'Arts/Design',
                'Government',                     'Other',
 'Engineering/Manufacturing',                 'Marketing',
              'Retail/Sales',                   'Biotech',
            'Transportation',               'Real Estate',
               'Agriculture',                         nan,
                       'Vet',               'Hospitality',
   'Contact Center Services']
Length: 23, dtype: str



industry
Information Technology       4776
Education                    3433
Nonprofit                    2430
Health                       2419
Finance                      2369
Other                        2219
Engineering/Manufacturing    2204
Government                   1942
Marketing                    1157
Legal                        1149
Media/Entertainment          1124
Business/Consulting           900
Retail/Sales                  523
Energy                        427
Arts/Design                   370
Transportation                310
Agriculture                   140
Biotech                        94
Hospitality                    55
Real Estate                    54
Vet                            27
Contact Center Services        17
Name: count, dtype: int64

## job_title

In [255]:
print_unique_summary(df, "job_title")
print("Total NULL Values:", df["job_title"].isnull().sum())

#There are 14434 unique values. Same approach to industry we used keyword matching data cleaning approach.
# 5 null values and will be included in Others

#after data cleaning 49 unique values

Total Unique Count (nunique): 14434
Unique Values (unique): <StringArray>
[      'Research and Instruction Librarian',
 'Change & Internal Communications Manager',
                     'Marketing Specialist',
                          'Program Manager',
                       'Accounting Manager',
           'Scholarly Publishing Librarian',
                     'Publishing Assistant',
                                'Librarian',
                          'Systems Analyst',
                        'Senior Accountant',
 ...
                'Director and Screenwriter',
                               'goashidfhu',
                 'Data and Insight Analyst',
  'Data Analytics and Reliability Engineer',
               'System programming analyst',
                                        'j',
                            'eye bank tech',
    ' Lead Messaging Operations Specialist',
            'Treasury Reporting Accountant',
                   'purchasing coordinator']
Length: 14435, dtype:

In [256]:
job_title_mapping = {
    # Engineering / Tech
    "Software Engineer": r"software engineer|swe\b|backend|frontend|full.?stack",
    "Data Scientist/Analyst": r"data scientist|data analyst|data engineer",
    "IT/Systems": r"\bit\b|systems admin|network admin|help.?desk|sysadmin",
    "Product Manager": r"product manager|product owner",
    "Project Manager": r"project manager|program manager|\bpmp?\b",
    "UX/UI Designer": r"ux|ui designer|user experience|user interface",
    "QA/Test Engineer": r"qa\b|quality assurance|test engineer",
    "DevOps/Cloud": r"devops|cloud engineer|site reliability|\bsre\b",

    # Management / Leadership
    "Executive (C-Suite)": r"\bceo\b|\bcfo\b|\bcoo\b|\bcto\b|chief .* officer|president",
    "Director": r"\bdirector\b",
    "VP/Senior Leadership": r"\bvp\b|vice president|senior vice president",
    "Manager (General)": r"\bmanager\b",
    "Supervisor": r"supervisor",
    "Team Lead": r"team lead|\blead\b",

    # Admin / Office
    "Administrative Assistant": r"administrative assistant|admin assistant|office assistant",
    "Executive Assistant": r"executive assistant|\bea\b",
    "Office Manager": r"office manager",
    "Receptionist": r"receptionist|front desk",
    "Coordinator": r"coordinator",

    # Finance / Accounting
    "Accountant": r"accountant|accounting",
    "Financial Analyst": r"financial analyst|finance analyst",
    "Bookkeeper": r"bookkeep",
    "Auditor": r"auditor|audit",
    "Controller": r"controller",

    # HR
    "HR Generalist/Manager": r"human resources|\bhr\b",
    "Recruiter": r"recruiter|talent acquisition",

    # Sales / Marketing
    "Sales Representative": r"sales rep|sales associate|account executive",
    "Sales Manager": r"sales manager",
    "Marketing Manager": r"marketing manager|marketing specialist",
    "Content/Copywriter": r"content writer|copywriter|content creator",
    "Social Media": r"social media",

    # Education
    "Teacher (K-12)": r"teacher|k-12|elementary|middle school|high school",
    "Professor/Faculty": r"professor|faculty|lecturer",
    "Teaching Assistant": r"teaching assistant|\bta\b",
    "Principal/Administrator (Edu)": r"principal|school administrator|dean",

    # Healthcare
    "Nurse": r"\brn\b|nurse",
    "Physician": r"physician|\bmd\b|doctor",
    "Therapist/Counselor": r"therapist|counselor|psycholog",
    "Medical Assistant": r"medical assistant",
    "Pharmacist": r"pharmacist",

    # Legal
    "Lawyer/Attorney": r"lawyer|attorney",
    "Paralegal": r"paralegal|legal assistant",

    # Customer / Support
    "Customer Service": r"customer service|customer support|client service",
    "Consultant": r"consultant",

    # Creative
    "Graphic Designer": r"graphic designer",
    "Writer/Editor": r"writer|editor|journalist",

    # Skilled trades / Other
    "Engineer (Non-software)": r"engineer",  # broad catch-all for civil/mechanical/electrical etc
    "Analyst (General)": r"analyst",
    "Librarian": r"librarian",
    "Social Worker": r"social worker",
}

df_temp["job_title"] = df_temp["job_title"].apply(lambda x: map_category(x, job_title_mapping))

In [257]:
df_temp["job_title"].nunique()

49

## annual_salary

In [258]:
print("Total NULL Values:", df["annual_salary"].isnull().sum())
print("annual_salary dtype:",df.annual_salary.dtype)
df.sample()

# The data type is String (dtype: str) and needs to convert to float
# Identify all string values that might contains (,)
# 0 null values so no need to convert null to 0
# float64 after data cleaning

Total NULL Values: 0
annual_salary dtype: str


,age_range,industry,job_title,annual_salary,currency,country,current_years_experience,education,gender
719,55-64,Government and Public Administration,Chief Counsel,"253,300",USD,USA,31 - 40 years,PhD,Man


In [259]:
try:
    df_temp['annual_salary'] = df_temp['annual_salary'].astype(float)
    print("Conversion successful!")
except ValueError as e:
    print(f"{e}")
# could not convert string to float: '55,000'
# need to convert all string values in annual_salary

could not convert string to float: '55,000'


In [260]:
df_temp['annual_salary'] = pd.to_numeric(
    df_temp['annual_salary'].apply(lambda x: str(x).replace(',', '')), 
    errors='coerce'
)

In [261]:
try:
    df_temp['annual_salary'] = df_temp['annual_salary'].astype(float)
    print("Conversion successful!")
except ValueError as e:
    print(f"{e}")
# could not convert string to float: '55,000'
# need to convert all string values in annual_salary

Conversion successful!


In [262]:
print("annual_salary dtype:",df.annual_salary.dtype)
print("after cleaning:",df_temp.annual_salary.dtype)
df_temp.sample()

annual_salary dtype: str
after cleaning: float64


,age_range,industry,job_title,annual_salary,currency,country,current_years_experience,education,gender
11451,25-34,Other,Other,28000.0,USD,US,5-7 years,College degree,Woman


## currency

In [263]:
print_unique_summary(df, "currency")
print_null_sum(df, "currency")

# no duplicate categories, no need to do data cleaning
# Remove Other Currency

Total Unique Count (nunique): 11
Unique Values (unique): <StringArray>
[    'USD',     'GBP',     'CAD',     'EUR', 'AUD/NZD',   'Other',     'CHF',
     'ZAR',     'SEK',     'HKD',     'JPY']
Length: 11, dtype: str

Total NULL Values: 0


In [264]:
df_temp.drop(df_temp[df_temp['currency'] == 'Other'].index, inplace=True)

In [265]:
df_temp['currency'].value_counts()

currency
USD        23497
CAD         1677
GBP         1598
EUR          654
AUD/NZD      505
CHF           37
SEK           37
JPY           23
ZAR           17
HKD            4
Name: count, dtype: int64

In [266]:
df_temp.head(1)

,age_range,industry,job_title,annual_salary,currency,country,current_years_experience,education,gender
0,25-34,Education,Librarian,55000.0,USD,United States,5-7 years,Master's degree,Woman


## country

In [267]:
print_unique_summary(df_temp, "country")
print_null_sum(df_temp, "country")

Total Unique Count (nunique): 363
Unique Values (unique): <StringArray>
[  'United States',  'United Kingdom',              'US',             'USA',
          'Canada', 'United Kingdom ',             'usa',              'UK',
       'Scotland ',            'U.S.',
 ...
         'Myanmar',           'Burma',           'india',          'Italia',
   'Hong KongKong',         'Liberia',               '1',    'Nigeria + UK',
               nan,         'romania']
Length: 364, dtype: str

Total NULL Values: 1


In [268]:
df_temp['country'].value_counts()

country
United States     9034
USA               7956
US                2618
Canada            1575
United States      669
                  ... 
Hong KongKong        1
Liberia              1
1                    1
Nigeria + UK         1
romania              1
Name: count, Length: 363, dtype: int64

In [269]:
df_temp['country'] = df_temp['country'].str.strip() #Normalize casing

In [270]:
country_mapping = {
    "United States": r"united states|^usa$|^us$|u\.s\.a?\.?",
    "United Kingdom": r"united kingdom|^uk$|england|scotland|wales|britain",
    "Canada": r"canada",
    "Australia": r"australia",
    "Germany": r"germany",
    "India": r"india",
    "Ireland": r"ireland",
    "France": r"france",
    "Netherlands": r"netherlands|holland",
    "New Zealand": r"new zealand",
}

df_temp["country"] = df_temp["country"].apply(lambda x: map_category(x, country_mapping))

In [271]:
df_temp["country"].value_counts()

country
United States     23101
Canada             1686
United Kingdom     1579
Other               678
Australia           386
Germany             199
Ireland             128
New Zealand         124
Netherlands          89
France               69
India                 9
Name: count, dtype: int64

In [272]:
others = df_temp[df_temp["country"] == "Other"]["country"]
others.value_counts().head(50)

# Find out 1200+ Other Country
# And what future data cleaning should we do

country
Other    678
Name: count, dtype: int64

## current_years_experience


In [273]:
print_null_sum(df, "current_years_experience")
print_unique_summary(df, "current_years_experience")
df.sample()

# no overlapping years experience, no null values. No data cleaning needed

Total NULL Values: 0
Total Unique Count (nunique): 8
Unique Values (unique): <StringArray>
[       '5-7 years',      '2 - 4 years',    '21 - 30 years',
    '11 - 20 years',   '1 year or less',     '8 - 10 years',
    '31 - 40 years', '41 years or more']
Length: 8, dtype: str



,age_range,industry,job_title,annual_salary,currency,country,current_years_experience,education,gender
2543,35-44,Computing or Tech,Principal Test Engineer,"125,000",USD,United States,11 - 20 years,Master's degree,Man


## education

In [274]:
print_null_sum(df, "education")
print_unique_summary(df, "education")
df.sample()

# 241 null values, 6 unique values

Total NULL Values: 241
Total Unique Count (nunique): 6
Unique Values (unique): <StringArray>
[                   'Master's degree',                     'College degree',
                                'PhD',                                  nan,
                       'Some college',                        'High School',
 'Professional degree (MD, JD, etc.)']
Length: 7, dtype: str



,age_range,industry,job_title,annual_salary,currency,country,current_years_experience,education,gender
5299,45-54,Education (Higher Education),Administrative Assistant,"49,000",USD,United States,8 - 10 years,College degree,Woman


In [275]:
df_temp['education'].value_counts(dropna=False)

education
College degree                        13486
Master's degree                        8858
Some college                           2081
PhD                                    1421
Professional degree (MD, JD, etc.)     1319
High School                             648
NaN                                     236
Name: count, dtype: int64

In [276]:
df_temp['education'].isnull().sum()

np.int64(236)

In [277]:
df_temp['education'].str.strip().str.lower().value_counts()

education
college degree                        13486
master's degree                        8858
some college                           2081
phd                                    1421
professional degree (md, jd, etc.)     1319
high school                             648
Name: count, dtype: int64

In [278]:
df_temp['education'].isnull().sum()

np.int64(236)

## gender

In [279]:
print_null_sum(df, "gender")
print_unique_summary(df, "gender")
df.sample()

# 186 null values, 6 unique
# convert nonbinary, other or prefer, prefer not to answer and null values to Prefer not to naswer

Total NULL Values: 186
Total Unique Count (nunique): 5
Unique Values (unique): <StringArray>
[                        'Woman',                    'Non-binary',
                           'Man',                             nan,
 'Other or prefer not to answer',          'Prefer not to answer']
Length: 6, dtype: str



,age_range,industry,job_title,annual_salary,currency,country,current_years_experience,education,gender
4528,25-34,Computing or Tech,Senior Content Strategist,"127,000",USD,USA,5-7 years,College degree,Woman


In [280]:
df_temp.loc[
    df_temp['gender'].isin(['Non-binary', 'Other or prefer not to answer']),
    'gender'
] = 'Prefer not to answer'

df_temp['gender'] = df_temp['gender'].fillna('Prefer not to answer')

In [281]:
df_temp['gender'].value_counts(dropna=False)

gender
Woman                   21319
Man                      5505
Prefer not to answer     1225
Name: count, dtype: int64

# Applying Data Cleaning
<b>Description</b>:
- Applying Data Cleaning to PROD/Actual Data Frame

## Industry
**Status:** n/a
- n/a
- n/a

In [282]:
# Industry
industry_mapping = {
    "Contact Center Services": r"\bcall\b|contact|customer service|\bbpo\b|business process|customer",
    "Education": r"teacher|educ|school|university|acade|phd|ed ?tech",
    "Health": r"health|medical|hospital|nursing|pharma|\bcare\b|psycho",
    "Biotech": r"biotech|bitech|pharma",
    "Finance": r"bank|finance|accounting|insurance|investment",
    "Legal": r"law|legal|attorney",
    "Government": r"government|public admin|civil service|military|govt",
    "Nonprofit": r"nonprofit|non-profit|ngo|charity",
    "Marketing": r"market|adverti|public relations",
    "Retail/Sales": r"retail|\bsale\b|e-commerce",
    "Engineering/Manufacturing": r"manufactur|engineer|construction|industrial",
    "Media/Entertainment": r"media|publish|journal|entertainment|film|music",
    "Hospitality": r"hospitality|restaurant|hotel|\bfood\b|\bchild\b|coffee",
    "Arts/Design": r"\bart\b|design|creative",
    "Real Estate": r"real estate|property",
    "Transportation": r"transport|logistics|shipping|airline",
    "Agriculture": r"agricult|\bfarm\b",
    "Energy": r"energy|\boil\b|\bgas\b|utilit",
    "Insurance": r"insurance",
    "Vet": r"vet|animal|\bcat\b|\bdog\b",
    "Business/Consulting": r"business|consult",
    "Information Technology": r"software|\bit\b|comput|\bdata\b|technology|information|cyber",
}

df["industry"] = df["industry"].apply(lambda x: map_category(x, industry_mapping))


## job_title
**Status:** n/a
- n/a
- n/a

In [283]:
job_title_mapping = {
    # Engineering / Tech
    "Software Engineer": r"software engineer|swe\b|backend|frontend|full.?stack",
    "Data Scientist/Analyst": r"data scientist|data analyst|data engineer",
    "IT/Systems": r"\bit\b|systems admin|network admin|help.?desk|sysadmin",
    "Product Manager": r"product manager|product owner",
    "Project Manager": r"project manager|program manager|\bpmp?\b",
    "UX/UI Designer": r"ux|ui designer|user experience|user interface",
    "QA/Test Engineer": r"qa\b|quality assurance|test engineer",
    "DevOps/Cloud": r"devops|cloud engineer|site reliability|\bsre\b",

    # Management / Leadership
    "Executive (C-Suite)": r"\bceo\b|\bcfo\b|\bcoo\b|\bcto\b|chief .* officer|president",
    "Director": r"\bdirector\b",
    "VP/Senior Leadership": r"\bvp\b|vice president|senior vice president",
    "Manager (General)": r"\bmanager\b",
    "Supervisor": r"supervisor",
    "Team Lead": r"team lead|\blead\b",

    # Admin / Office
    "Administrative Assistant": r"administrative assistant|admin assistant|office assistant",
    "Executive Assistant": r"executive assistant|\bea\b",
    "Office Manager": r"office manager",
    "Receptionist": r"receptionist|front desk",
    "Coordinator": r"coordinator",

    # Finance / Accounting
    "Accountant": r"accountant|accounting",
    "Financial Analyst": r"financial analyst|finance analyst",
    "Bookkeeper": r"bookkeep",
    "Auditor": r"auditor|audit",
    "Controller": r"controller",

    # HR
    "HR Generalist/Manager": r"human resources|\bhr\b",
    "Recruiter": r"recruiter|talent acquisition",

    # Sales / Marketing
    "Sales Representative": r"sales rep|sales associate|account executive",
    "Sales Manager": r"sales manager",
    "Marketing Manager": r"marketing manager|marketing specialist",
    "Content/Copywriter": r"content writer|copywriter|content creator",
    "Social Media": r"social media",

    # Education
    "Teacher (K-12)": r"teacher|k-12|elementary|middle school|high school",
    "Professor/Faculty": r"professor|faculty|lecturer",
    "Teaching Assistant": r"teaching assistant|\bta\b",
    "Principal/Administrator (Edu)": r"principal|school administrator|dean",

    # Healthcare
    "Nurse": r"\brn\b|nurse",
    "Physician": r"physician|\bmd\b|doctor",
    "Therapist/Counselor": r"therapist|counselor|psycholog",
    "Medical Assistant": r"medical assistant",
    "Pharmacist": r"pharmacist",

    # Legal
    "Lawyer/Attorney": r"lawyer|attorney",
    "Paralegal": r"paralegal|legal assistant",

    # Customer / Support
    "Customer Service": r"customer service|customer support|client service",
    "Consultant": r"consultant",

    # Creative
    "Graphic Designer": r"graphic designer",
    "Writer/Editor": r"writer|editor|journalist",

    # Skilled trades / Other
    "Engineer (Non-software)": r"engineer",  # broad catch-all for civil/mechanical/electrical etc
    "Analyst (General)": r"analyst",
    "Librarian": r"librarian",
    "Social Worker": r"social worker",
}

df["job_title"] = df["job_title"].apply(lambda x: map_category(x, job_title_mapping))

## annual_salary
**Status:** n/a
- n/a
- n/a

In [284]:
df['annual_salary'] = pd.to_numeric(
    df['annual_salary'].apply(lambda x: str(x).replace(',', '')), 
    errors='coerce'
)

## currency
**Status:** n/a
- n/a
- n/a

In [285]:
df.drop(df[df['currency'] == 'Other'].index, inplace=True)

## country
**Status:** n/a
- n/a
- n/a

In [286]:
country_mapping = {
    "United States": r"united states|^usa$|^us$|u\.s\.a?\.?",
    "United Kingdom": r"united kingdom|^uk$|england|scotland|wales|britain",
    "Canada": r"canada",
    "Australia": r"australia",
    "Germany": r"germany",
    "India": r"india",
    "Ireland": r"ireland",
    "France": r"france",
    "Netherlands": r"netherlands|holland",
    "New Zealand": r"new zealand",
}

df["country"] = df["country"].apply(lambda x: map_category(x, country_mapping))

## gender
**Status:** n/a
- n/a
- n/a

In [287]:
df.loc[
    df['gender'].isin(['Non-binary', 'Other or prefer not to answer']),
    'gender'
] = 'Prefer not to answer'

df['gender'] = df['gender'].fillna('Prefer not to answer')

# Cleaned Data

In [288]:
df.to_csv('../Dataset/salary_survey_cleaned.csv', index=False)